# CCHS 2022 Diabetes Classification — Controlled Experiment Setup

This notebook defines and verifies the controlled experimental framework for the CCHS 2022 diabetes classification study.

It establishes the feature conditions, eligible modelling populations, fixed train/validation/test splits, CCHS special-value handling, harmonised BMI representation, and preprocessing rules used by the subsequent experiment notebooks.

The actual model training, evaluation, and failure analysis are performed separately for each controlled experiment.

In [44]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [45]:
pumf = pd.read_csv("pumf_cchs.csv")
bootstrap = pd.read_csv("cchs_escc_bsw.csv")

print("PUMF shape:", pumf.shape)
print("Bootstrap-weight file shape:", bootstrap.shape)

PUMF shape: (67079, 255)
Bootstrap-weight file shape: (67079, 1002)


In [46]:
print("\nPUMF columns:", len(pumf.columns))
print("Bootstrap-weight columns:", len(bootstrap.columns))


PUMF columns: 255
Bootstrap-weight columns: 1002


In [47]:
print(pumf["CCC_05"].value_counts(dropna=False).sort_index())

CCC_05
1     5994
2    60248
9      837
Name: count, dtype: int64


In [48]:
target_counts = pumf["CCC_05"].value_counts(dropna=False).sort_index()

print("Diabetes = 1:", target_counts.get(1, 0))
print("No diabetes = 2:", target_counts.get(2, 0))
print("Not stated = 9:", target_counts.get(9, 0))

Diabetes = 1: 5994
No diabetes = 2: 60248
Not stated = 9: 837


In [49]:
model_data = pumf[pumf["CCC_05"].isin([1, 2])].copy()

model_data["target"] = (model_data["CCC_05"] == 1).astype(int)

print("Modelling population:", model_data.shape[0])
print(model_data["target"].value_counts())

Modelling population: 66242
target
0    60248
1     5994
Name: count, dtype: int64


## Experimental Feature Conditions

The feature sets below were defined before model fitting so that the experiments remain controlled and comparable.

The main experiments use the same 12+ modelling population. Population-restricted experiments are defined separately for respondents who meet the relevant CCHS variable universe.

The original baseline is preserved as a historical reference, while the remaining conditions represent progressively different feature-selection decisions.

In [50]:
# Frozen feature conditions for Notebook 02

FEATURE_SETS = {
    "A_Baseline": [
        "DHHGAGE", "DHH_SEX", "GEOGPRV", "DHHDGHSZ", "EDDVH3",
        "LSM_01", "HWTDGISW", "WTP_50", "SMKDVSTY",
        "FSCDVHF2", "INCDGHH"
    ],

    "B_Preliminary": [
        "ALCDVTTM", "DHHDGHSZ", "DHHGAGE", "DHH_SEX", "EDDVH3",
        "FSCDVHF2", "GEOGPRV", "BMI_CLASS", "INCDGHH",
        "ECV_05", "SDCDGIMM"
    ],

    "C_Literature_Core": [
        "DHHGAGE", "DHH_SEX", "EDDVH3",
        "BMI_CLASS", "INCDGHH", "SDCDGIMM", "GEOGPRV"
    ],

    "D1_Core_Hypertension": [
        "DHHGAGE", "DHH_SEX", "EDDVH3",
        "BMI_CLASS", "INCDGHH", "SDCDGIMM", "GEOGPRV",
        "CCC_80"
    ],

    "D2_Core_Hypertension_Cholesterol": [
        "DHHGAGE", "DHH_SEX", "EDDVH3",
        "BMI_CLASS", "INCDGHH", "SDCDGIMM", "GEOGPRV",
        "CCC_80", "CCC_90"
    ],

    "D3_35plus_Clinical": [
        "DHHGAGE", "DHH_SEX", "EDDVH3",
        "HWTDGISW", "INCDGHH", "SDCDGIMM", "GEOGPRV",
        "CCC_80", "CCC_90", "CCCDGCAR"
    ],

    "E1_Adult_Smoking": [
        "DHHGAGE", "DHH_SEX", "EDDVH3",
        "HWTDGISW", "INCDGHH", "SDCDGIMM", "GEOGPRV",
        "SMKDVSTY"
    ],

    "E2_Adult_Physical_Activity": [
        "DHHGAGE", "DHH_SEX", "EDDVH3",
        "HWTDGISW", "INCDGHH", "SDCDGIMM", "GEOGPRV",
        "PAADVMVA"
    ]
}

for name, features in FEATURE_SETS.items():
    print(f"{name}: {len(features)} features")
    print(features)
    print()

A_Baseline: 11 features
['DHHGAGE', 'DHH_SEX', 'GEOGPRV', 'DHHDGHSZ', 'EDDVH3', 'LSM_01', 'HWTDGISW', 'WTP_50', 'SMKDVSTY', 'FSCDVHF2', 'INCDGHH']

B_Preliminary: 11 features
['ALCDVTTM', 'DHHDGHSZ', 'DHHGAGE', 'DHH_SEX', 'EDDVH3', 'FSCDVHF2', 'GEOGPRV', 'BMI_CLASS', 'INCDGHH', 'ECV_05', 'SDCDGIMM']

C_Literature_Core: 7 features
['DHHGAGE', 'DHH_SEX', 'EDDVH3', 'BMI_CLASS', 'INCDGHH', 'SDCDGIMM', 'GEOGPRV']

D1_Core_Hypertension: 8 features
['DHHGAGE', 'DHH_SEX', 'EDDVH3', 'BMI_CLASS', 'INCDGHH', 'SDCDGIMM', 'GEOGPRV', 'CCC_80']

D2_Core_Hypertension_Cholesterol: 9 features
['DHHGAGE', 'DHH_SEX', 'EDDVH3', 'BMI_CLASS', 'INCDGHH', 'SDCDGIMM', 'GEOGPRV', 'CCC_80', 'CCC_90']

D3_35plus_Clinical: 10 features
['DHHGAGE', 'DHH_SEX', 'EDDVH3', 'HWTDGISW', 'INCDGHH', 'SDCDGIMM', 'GEOGPRV', 'CCC_80', 'CCC_90', 'CCCDGCAR']

E1_Adult_Smoking: 8 features
['DHHGAGE', 'DHH_SEX', 'EDDVH3', 'HWTDGISW', 'INCDGHH', 'SDCDGIMM', 'GEOGPRV', 'SMKDVSTY']

E2_Adult_Physical_Activity: 8 features
['DHHGAGE', '

## Modelling Populations

Most experiments use the general CCHS modelling population aged 12 and older after excluding unusable diabetes-target responses.

Some feature conditions require a restricted population because of the CCHS universe of the variables being tested. These populations are defined explicitly rather than treating structural skips as ordinary missing observations.

In [51]:
print("DHHGAGE value counts:")
print(model_data["DHHGAGE"].value_counts(dropna=False).sort_index())

print("\nDOPAA value counts:")
print(model_data["DOPAA"].value_counts(dropna=False).sort_index())

DHHGAGE value counts:
DHHGAGE
1     3744
2    10058
3    12616
4    16041
5    23783
Name: count, dtype: int64

DOPAA value counts:
DOPAA
0    64323
1     1919
Name: count, dtype: int64


In [52]:
print("CCCDGCAR value counts:")
print(model_data["CCCDGCAR"].value_counts(dropna=False).sort_index())

CCCDGCAR value counts:
CCCDGCAR
1     5532
2    45420
6    13802
9     1488
Name: count, dtype: int64


In [53]:
# Define the eligible population for each experiment

general_idx = model_data.index

adult_idx = model_data.index[
    model_data["DHHGAGE"].isin([2, 3, 4, 5])
]

age35_idx = model_data.index[
    model_data["DHHGAGE"].isin([3, 4, 5])
]

paa_idx = model_data.index[
    model_data["DHHGAGE"].isin([2, 3, 4, 5]) &
    (model_data["DOPAA"] == 1)
]

POPULATIONS = {
    "A_Baseline": general_idx,
    "B_Preliminary": general_idx,
    "C_Literature_Core": general_idx,
    "D1_Core_Hypertension": general_idx,
    "D2_Core_Hypertension_Cholesterol": general_idx,
    "D3_35plus_Clinical": age35_idx,
    "E1_Adult_Smoking": adult_idx,
    "E2_Adult_Physical_Activity": paa_idx
}

for name, idx in POPULATIONS.items():
    print(f"{name}: {len(idx)} respondents")

A_Baseline: 66242 respondents
B_Preliminary: 66242 respondents
C_Literature_Core: 66242 respondents
D1_Core_Hypertension: 66242 respondents
D2_Core_Hypertension_Cholesterol: 66242 respondents
D3_35plus_Clinical: 52440 respondents
E1_Adult_Smoking: 62498 respondents
E2_Adult_Physical_Activity: 1817 respondents


In [54]:
print("Adult population:", len(adult_idx))
print("35+ population:", len(age35_idx))
print("Adult PAA population:", len(paa_idx))

Adult population: 62498
35+ population: 52440
Adult PAA population: 1817


In [55]:
# Check target distribution within each modelling population

for name, idx in POPULATIONS.items():
    y_pop = model_data.loc[idx, "target"]

    print(f"\n{name}")
    print(f"Respondents: {len(y_pop)}")
    print(y_pop.value_counts().sort_index())
    print(f"Diabetes prevalence: {y_pop.mean():.3%}")


A_Baseline
Respondents: 66242
target
0    60248
1     5994
Name: count, dtype: int64
Diabetes prevalence: 9.049%

B_Preliminary
Respondents: 66242
target
0    60248
1     5994
Name: count, dtype: int64
Diabetes prevalence: 9.049%

C_Literature_Core
Respondents: 66242
target
0    60248
1     5994
Name: count, dtype: int64
Diabetes prevalence: 9.049%

D1_Core_Hypertension
Respondents: 66242
target
0    60248
1     5994
Name: count, dtype: int64
Diabetes prevalence: 9.049%

D2_Core_Hypertension_Cholesterol
Respondents: 66242
target
0    60248
1     5994
Name: count, dtype: int64
Diabetes prevalence: 9.049%

D3_35plus_Clinical
Respondents: 52440
target
0    46547
1     5893
Name: count, dtype: int64
Diabetes prevalence: 11.238%

E1_Adult_Smoking
Respondents: 62498
target
0    56527
1     5971
Name: count, dtype: int64
Diabetes prevalence: 9.554%

E2_Adult_Physical_Activity
Respondents: 1817
target
0    1711
1     106
Name: count, dtype: int64
Diabetes prevalence: 5.834%


In [56]:
# Confirm that the restricted populations are subsets of the general modelling population

assert set(adult_idx).issubset(set(general_idx))
assert set(age35_idx).issubset(set(general_idx))
assert set(paa_idx).issubset(set(adult_idx))

print("Population hierarchy checks passed.")

Population hierarchy checks passed.


## Train / Validation / Test Splits

A fixed stratified split is created for each modelling population.

For experiments using the same population, the same split is reused so that feature-set and model comparisons are made on the same observations.

Population-restricted experiments receive their own fixed split within the corresponding eligible population.

The test set remains untouched during feature selection, model selection, and threshold selection.

In [57]:
RANDOM_STATE = 42

def make_split(index, test_size=0.20, validation_size=0.20):
    """Create a stratified train/validation/test split of respondent indices."""

    population = model_data.loc[index]

    train_val_idx, test_idx = train_test_split(
        population.index,
        test_size=test_size,
        stratify=population["target"],
        random_state=RANDOM_STATE
    )

    train_idx, val_idx = train_test_split(
        train_val_idx,
        test_size=validation_size,
        stratify=model_data.loc[train_val_idx, "target"],
        random_state=RANDOM_STATE
    )

    return train_idx, val_idx, test_idx

## Split Configuration

The random seed is fixed at 42 so that the same modelling population produces the same train, validation, and test assignments throughout the notebook.

Population-restricted experiments use the same procedure within their eligible population.

In [58]:
SPLITS = {}

for name, idx in POPULATIONS.items():
    train_idx, val_idx, test_idx = make_split(idx)

    SPLITS[name] = {
        "train": train_idx,
        "validation": val_idx,
        "test": test_idx
    }

    print(
        f"{name}: "
        f"train={len(train_idx)}, "
        f"validation={len(val_idx)}, "
        f"test={len(test_idx)}"
    )

A_Baseline: train=42394, validation=10599, test=13249
B_Preliminary: train=42394, validation=10599, test=13249
C_Literature_Core: train=42394, validation=10599, test=13249
D1_Core_Hypertension: train=42394, validation=10599, test=13249
D2_Core_Hypertension_Cholesterol: train=42394, validation=10599, test=13249
D3_35plus_Clinical: train=33561, validation=8391, test=10488
E1_Adult_Smoking: train=39998, validation=10000, test=12500
E2_Adult_Physical_Activity: train=1162, validation=291, test=364


In [59]:
for name, split in SPLITS.items():
    train_y = model_data.loc[split["train"], "target"]
    val_y = model_data.loc[split["validation"], "target"]
    test_y = model_data.loc[split["test"], "target"]

    print(f"\n{name}")
    print(
        f"Train: {len(train_y)} ({train_y.mean():.3%} positive) | "
        f"Validation: {len(val_y)} ({val_y.mean():.3%}) | "
        f"Test: {len(test_y)} ({test_y.mean():.3%})"
    )


A_Baseline
Train: 42394 (9.048% positive) | Validation: 10599 (9.048%) | Test: 13249 (9.050%)

B_Preliminary
Train: 42394 (9.048% positive) | Validation: 10599 (9.048%) | Test: 13249 (9.050%)

C_Literature_Core
Train: 42394 (9.048% positive) | Validation: 10599 (9.048%) | Test: 13249 (9.050%)

D1_Core_Hypertension
Train: 42394 (9.048% positive) | Validation: 10599 (9.048%) | Test: 13249 (9.050%)

D2_Core_Hypertension_Cholesterol
Train: 42394 (9.048% positive) | Validation: 10599 (9.048%) | Test: 13249 (9.050%)

D3_35plus_Clinical
Train: 33561 (11.236% positive) | Validation: 8391 (11.238%) | Test: 10488 (11.241%)

E1_Adult_Smoking
Train: 39998 (9.555% positive) | Validation: 10000 (9.550%) | Test: 12500 (9.552%)

E2_Adult_Physical_Activity
Train: 1162 (5.852% positive) | Validation: 291 (5.842%) | Test: 364 (5.769%)


In [60]:
# Create a modelling copy with a harmonized BMI feature

model_data = model_data.copy()

# Start with missing values
model_data["BMI_CLASS"] = np.nan

# Ages 12–17: use the youth BMI classification
youth_mask = model_data["DHHGAGE"] == 1
model_data.loc[youth_mask, "BMI_CLASS"] = model_data.loc[youth_mask, "HWTDGWHO"]

# Ages 18+: use the adult BMI classification
adult_mask = model_data["DHHGAGE"].isin([2, 3, 4, 5])
model_data.loc[adult_mask, "BMI_CLASS"] = model_data.loc[adult_mask, "HWTDGISW"]

print(model_data["BMI_CLASS"].value_counts(dropna=False).sort_index())

BMI_CLASS
1.0    26053
2.0    37045
6.0       32
9.0     3112
Name: count, dtype: int64


In [61]:
# Inspect data types and observed values for the main feature sets

inspection_features = sorted(
    set(
        FEATURE_SETS["A_Baseline"]
        + FEATURE_SETS["B_Preliminary"]
        + FEATURE_SETS["C_Literature_Core"]
    )
)

feature_inspection = pd.DataFrame({
    "dtype": model_data[inspection_features].dtypes.astype(str),
    "unique_values": [
        sorted(model_data[col].dropna().unique().tolist())
        for col in inspection_features
    ],
    "n_unique": [
        model_data[col].nunique(dropna=True)
        for col in inspection_features
    ]
})

feature_inspection

,dtype,unique_values,n_unique
ALCDVTTM,int64,"[1, 2, 3, 9]",4
BMI_CLASS,float64,"[1.0, 2.0, 6.0, 9.0]",4
DHHDGHSZ,int64,"[1, 2, 9]",3
DHHGAGE,int64,"[1, 2, 3, 4, 5]",5
DHH_SEX,int64,"[1, 2]",2
ECV_05,int64,"[1, 2, 9]",3
EDDVH3,int64,"[1, 2, 3, 9]",4
FSCDVHF2,int64,"[0, 1, 2, 3, 9]",5
GEOGPRV,int64,"[10, 11, 12, 13, 24, 35, 46, 47, 48, 59, 60]",11
HWTDGISW,int64,"[1, 2, 6, 9]",4


In [62]:
# Check the same variables in the training portion of the general population

general_train_idx = SPLITS["A_Baseline"]["train"]

train_inspection = pd.DataFrame({
    "dtype": model_data.loc[general_train_idx, inspection_features]
        .dtypes.astype(str),
    "n_unique_train": [
        model_data.loc[general_train_idx, col].nunique(dropna=True)
        for col in inspection_features
    ]
})

train_inspection

,dtype,n_unique_train
ALCDVTTM,int64,4
BMI_CLASS,float64,4
DHHDGHSZ,int64,3
DHHGAGE,int64,5
DHH_SEX,int64,2
ECV_05,int64,3
EDDVH3,int64,4
FSCDVHF2,int64,5
GEOGPRV,int64,11
HWTDGISW,int64,4


## Feature Preprocessing Rules

The selected CCHS variables are integer-coded categorical variables rather than continuous measurements.

CCHS special codes are handled according to the documentation. Age-specific BMI variables require separate treatment because each variable applies to a different age group.

For the broadly applicable 12+ experiments, the adult and youth BMI classifications will therefore be combined into one BMI representation. This avoids treating an age-based valid skip as an ordinary missing observation.

In [63]:
print("BMI_CLASS by age group:")
print(
    pd.crosstab(
        model_data["DHHGAGE"],
        model_data["BMI_CLASS"],
        dropna=False
    )
)

BMI_CLASS by age group:
BMI_CLASS   1.0    2.0  6.0  9.0
DHHGAGE                         
1          2444    965    0  335
2          4861   4560   10  627
3          4487   7517    3  609
4          5257  10116    8  660
5          9004  13887   11  881


In [64]:
print(
    model_data[
        ["DHHGAGE", "HWTDGISW", "HWTDGWHO", "BMI_CLASS"]
    ].head(20)
)

    DHHGAGE  HWTDGISW  HWTDGWHO  BMI_CLASS
0         5         1         6        1.0
1         4         2         6        2.0
2         1         6         1        1.0
3         2         1         6        1.0
4         3         1         6        1.0
5         5         2         6        2.0
6         3         2         6        2.0
7         5         2         6        2.0
8         2         2         6        2.0
9         3         2         6        2.0
10        5         2         6        2.0
11        5         2         6        2.0
12        3         2         6        2.0
13        5         2         6        2.0
14        5         2         6        2.0
15        5         2         6        2.0
16        4         1         6        1.0
17        5         1         6        1.0
18        2         2         6        2.0
19        3         1         6        1.0


## CCHS Special-Value Handling

CCHS special codes are interpreted according to the documentation rather than treated as ordinary category levels.

For the variables used in each experiment, population-exclusion/valid-skip and not-stated codes are converted to missing values before model fitting. The conversion is deterministic; any subsequent imputation is learned from the training data only.

In [65]:
# CCHS special codes used by the selected feature variables

SPECIAL_CODES = {
    "DHHDGHSZ": [9],
    "DHHGAGE": [],
    "DHH_SEX": [],
    "EDDVH3": [9],
    "FSCDVHF2": [9],
    "GEOGPRV": [],
    "INCDGHH": [9],
    "ALCDVTTM": [9],
    "ECV_05": [9],
    "SDCDGIMM": [9],
    "LSM_01": [99],
    "WTP_50": [9],
    "SMKDVSTY": [96, 99],
    "HWTDGISW": [6, 9],
    "HWTDGWHO": [6, 9],
    "CCC_80": [9],
    "CCC_90": [9],
    "CCCDGCAR": [6, 9],
    "PAADVMVA": [6, 9],
    "BMI_CLASS": [6, 9]
}

def apply_special_codes(df, special_codes):
    """Convert documented CCHS special codes to NaN."""

    result = df.copy()

    for column, codes in special_codes.items():
        if column in result.columns:
            result[column] = result[column].replace(codes, np.nan)

    return result

## Preprocessing

The selected CCHS variables are prepared differently according to their measurement type.

Categorical variables are imputed and one-hot encoded. Numeric variables are imputed and standardized.

Preprocessing is fitted using the training split only and then applied to the validation and test splits.

In [66]:
# Apply the defined CCHS special-value handling
clean_model_data = apply_special_codes(model_data, SPECIAL_CODES)

# Numeric variables for each experiment
NUMERIC_FEATURES = {
    "A_Baseline": ["LSM_01"],
    "B_Preliminary": [],
    "C_Literature_Core": [],
    "D1_Core_Hypertension": [],
    "D2_Core_Hypertension_Cholesterol": [],
    "D3_35plus_Clinical": [],
    "E1_Adult_Smoking": [],
    "E2_Adult_Physical_Activity": ["PAADVMVA"]
}

def build_preprocessor(experiment_name):
    features = FEATURE_SETS[experiment_name]
    numeric_features = NUMERIC_FEATURES[experiment_name]
    categorical_features = [
        feature for feature in features
        if feature not in numeric_features
    ]

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])

    return preprocessor

In [67]:
# Test the preprocessing pipeline on the baseline experiment

experiment = "A_Baseline"

features = FEATURE_SETS[experiment]
train_idx = SPLITS[experiment]["train"]
val_idx = SPLITS[experiment]["validation"]
test_idx = SPLITS[experiment]["test"]

X_train = clean_model_data.loc[train_idx, features]
X_val = clean_model_data.loc[val_idx, features]
X_test = clean_model_data.loc[test_idx, features]

preprocessor = build_preprocessor(experiment)

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Raw training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Raw training shape: (42394, 11)
Processed training shape: (42394, 44)
Processed validation shape: (10599, 44)
Processed test shape: (13249, 44)


In [68]:
y_train = clean_model_data.loc[train_idx, "target"]
y_val = clean_model_data.loc[val_idx, "target"]
y_test = clean_model_data.loc[test_idx, "target"]

print("Training:", X_train_processed.shape, y_train.shape)
print("Validation:", X_val_processed.shape, y_val.shape)
print("Test:", X_test_processed.shape, y_test.shape)

print("\nTraining diabetes prevalence:", y_train.mean())
print("Validation diabetes prevalence:", y_val.mean())
print("Test diabetes prevalence:", y_test.mean())

Training: (42394, 44) (42394,)
Validation: (10599, 44) (10599,)
Test: (13249, 44) (13249,)

Training diabetes prevalence: 0.09048450252394206
Validation diabetes prevalence: 0.09048023398433815
Test diabetes prevalence: 0.09049739602988904


## Notebook Status

The controlled experimental framework has been verified successfully.

The dataset, modelling populations, feature conditions, train/validation/test splits, harmonised BMI representation, CCHS special-value handling, and preprocessing pipeline are ready for the separate experiment notebooks.

Model training and evaluation are intentionally performed outside this setup notebook.